# 14 · Factor models on the initiation split: GRU vs gradient-boosted trees (E14)

New split (chosen from the price path only, before any model was run on it): **train** 2024-01-01 → 2025-01-30,
**val** 2025-02-01 → 2025-06-29, **test** 2025-07-01 → 2025-10-30, **stress_bear** 2025-11-01 → 2026-06-06, one
embargo day at each boundary. Everything below is fitted on train and chosen on val. **Test and stress_bear are
predicted and saved but not shown here.**

Three input versions (built locally by `initiation/prepare_init.py`):

| set | inputs | what |
|---|---|---|
| `legacy` | 101 | Alpha101 as in E4–E13 (`midpoint` used as the VWAP) |
| `fixed` | 101 | same formulas given the real VWAP from the 1-second klines |
| `fixed_plus` | 115 | `fixed` + 14 intra-minute features from the 1-second klines |

Two factor models, same targets (8 trend horizons, rltrade.labels) and the same output files, so the agent
can use either:

* **GRU**: E4's selected configuration (`gru`, mid 192, 2 layers, RMSprop, ...). Its hyperparameters were tuned
  on the old split, whose validation partly overlaps the new test. Only its *settings* come from there, not its
  weights. A re-tune on the new validation is a later step.
* **GBDT**: LightGBM, one model per horizon, on each input plus its 15- and 60-bar moving averages
  (`rltrade.rnn.gbdt`). Default settings fixed in advance, early stopping on val.

Selection rule (as E4): lowest mean validation MSE over seeds; IC and direction accuracy are reported too.

**Input (upload once, never commit):** `thesis_rl/initiation/initiation_prepared.zip` (the three folders of
`initiation/data/prepared/`). The notebook unzips it to the Colab disk. Runs resume: finished runs are skipped.

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/14-initiation"
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available(): print('WARNING: no GPU - Runtime > Change runtime type > T4 GPU')

In [ ]:
seeding.record_environment(L.report / "environment", "14_initiation_factors")

In [ ]:
import json, zipfile, shutil, numpy as np, pandas as pd
from pathlib import Path
try:
    import lightgbm
except ImportError:                                # preinstalled on Colab; install only if missing
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm"], check=True)
    import lightgbm
print("lightgbm", lightgbm.__version__)
from rltrade.rnn.train import RnnConfig, train_rnn
from rltrade.rnn.gbdt import GbdtConfig, train_gbdt

INIT = L.root / "initiation"                       # Drive: zip in, factors and report out
OUT, REP = INIT / "factors", INIT / "report"
for p in (OUT, REP): p.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/initiation_prepared") if "google.colab" in sys.modules else INIT / "prepared"
ZIP = INIT / "initiation_prepared.zip"
if not (LOCAL / "fixed_plus" / "meta.json").exists():
    assert ZIP.exists(), f"upload initiation_prepared.zip to {ZIP}"
    print("unzipping", ZIP, "->", LOCAL)
    with zipfile.ZipFile(ZIP) as z: z.extractall(LOCAL)
SETS = ("legacy", "fixed", "fixed_plus")
for s in SETS:
    m = json.loads((LOCAL / s / "meta.json").read_text())
    print(f"{s:11s} {m['n_inputs']:3d} inputs | dead on train {m['dead_inputs_on_train']} | "
          + " | ".join(f"{k} {v['start'][:10]}..{v['end'][:10]}" for k, v in m["splits"].items()))
SEEDS = [0, 1, 2]
# E4's selected GRU (thesis_rl/rnn/tuning/final/gru_s1/config.json), seed and device left free
GRU_CFG = dict(arch="gru", hp={"mid_dim": 192, "num_layers": 2, "dropout": 0.21851597689967073},
               seq_len=256, wup_dim=64, batch_size=128, optimizer="rmsprop", lr=0.000978618567468893,
               weight_decay=4.277083049962067e-05, clip_grad_norm=1.0, epoch=1024, max_minutes=60,
               valid_gap=50, patience=10, valid_windows=256, input_noise=0.01,
               label_noise=0.005102240537401404, skip_head=1024, amp=True)
def run_dir(model, s, seed): return OUT / f"{model}_{s}_s{seed}"

## GRU (GPU, about a minute per run)

In [ ]:
for seed in SEEDS:
    for s in SETS:
        d = run_dir("gru", s, seed)
        if (d / "metrics.json").exists(): print("done:", d.name); continue
        print("=====", d.name)
        train_rnn(RnnConfig(**GRU_CFG, seed=seed), LOCAL / s, d)

## Gradient-boosted trees (CPU: 2.3 min per run on 8 cores, expect roughly 5-10 min on Colab's 2)

In [ ]:
for seed in SEEDS:                 # seed 0 for every set first, so one full comparison exists early
    for s in SETS:
        d = run_dir("gbdt", s, seed)
        if (d / "metrics.json").exists(): print("done:", d.name); continue
        print("=====", d.name)
        train_gbdt(GbdtConfig(seed=seed), LOCAL / s, d)

## Comparison on validation (test and stress_bear stay hidden)

In [ ]:
rows = []
for d in sorted(OUT.glob("*_s*")):
    f = d / "metrics.json"
    if not f.exists(): continue
    m = json.loads(f.read_text())
    model, rest = d.name.split("_", 1); s, seed = rest.rsplit("_s", 1)
    for split in ("train", "val"):
        rows.append({"model": model, "set": s, "seed": int(seed), "split": split, "mse": m[split]["mse"],
                     "ic": m[split]["ic_mean"], "dir_acc": m[split]["dir_acc_mean"],
                     **{f"ic_h{k}": v for k, v in enumerate(m[split]["ic"])}})
runs = pd.DataFrame(rows)
runs.to_csv(REP / "e14_runs_train_val.csv", index=False)
val = runs[runs.split == "val"]
summ = (val.groupby(["model", "set"])[["mse", "ic", "dir_acc"]].agg(["mean", "std", "count"]))
summ.columns = [f"{a}_{b}" for a, b in summ.columns]
summ = summ.sort_values("mse_mean")
print(summ.round(4).to_string())
best = summ.index[0]
print("\nselected (lowest mean val MSE):", best)
per_h = val.groupby(["model", "set"])[[f"ic_h{k}" for k in range(8)]].mean().round(3)
print("\nvalidation IC per horizon (10, 20, 30, 60, 80, 100, 200, 400 bars):\n" + per_h.to_string())
tr = runs[runs.split == "train"].groupby(["model", "set"])[["mse", "ic"]].mean().add_prefix("train_")
print("\ntrain vs val (overfitting check):\n" + tr.join(summ[["mse_mean", "ic_mean"]]).round(4).to_string())
(REP / "e14_factor_models.md").write_text("# E14 factor models, validation (mean over seeds)\n\n"
    + summ.round(4).to_markdown() + "\n\nSelected: " + str(best) + "\n\nValidation IC per horizon\n\n"
    + per_h.to_markdown() + "\n", encoding="utf-8")
sel_model, sel_set = best
sel_seed = int(val[(val.model == sel_model) & (val.set == sel_set)].sort_values("mse").seed.iloc[0])
(INIT / "selected_factors.json").write_text(json.dumps({"model": sel_model, "set": sel_set, "seed": sel_seed,
    "factor_dir": str(run_dir(sel_model, sel_set, sel_seed)),
    "rule": "lowest mean validation MSE over seeds; best-validation seed of that model and set"}, indent=1))
print("wrote", INIT / "selected_factors.json")

## Which inputs the trees use (fixed_plus, mean over seeds)

In [ ]:
imps = [pd.read_csv(d / "importance_inputs.csv", index_col=0).gain_share for d in OUT.glob("gbdt_fixed_plus_s*")
        if (d / "importance_inputs.csv").exists()]
if imps:
    imp = pd.concat(imps, axis=1).mean(axis=1).sort_values(ascending=False)
    intra = {"ret_1m", "open_gap", "range_1m", "rv_1s", "rv_up_1s", "rv_down_1s", "max_abs_ret_1s", "ret_last10s",
             "close_pos", "close_vwap_gap", "flow_imbalance", "active_frac", "log_dollar_volume", "amihud_1s"}
    print(f"share of gain from the 14 intra-minute inputs: {imp[imp.index.isin(intra)].sum():.1%}")
    print(imp.head(25).round(4).to_string())
    imp.to_csv(REP / "e14_gbdt_importance_fixed_plus.csv")